In [ ]:
import sys
import community as louvain_community
import networkx as nx
import itertools
import infomap
from collections import Counter
import subprocess


sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/pipeline/cd_preprocessing.py'
%run '../../legal-data-clustering/legal_data_clustering/pipeline/cd_cluster.py'
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
%run 'common.py'

import altair.vega.v5 as alt
import cdlib
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm, colors
import matplotlib.patches as mpatches


In [ ]:
edge_filter_weight = 10 #

In [ ]:
%matplotlib inline

In [ ]:
light_reds_cm = colors.ListedColormap(
    (
#     (1.0                , 0.96078431372549022 , 0.94117647058823528),
    (0.99607843137254903, 0.8784313725490196  , 0.82352941176470584),
    (0.9882352941176471 , 0.73333333333333328 , 0.63137254901960782),
    (0.9882352941176471 , 0.5725490196078431  , 0.44705882352941179),
    (0.98431372549019602, 0.41568627450980394 , 0.29019607843137257),
    (0.93725490196078431, 0.23137254901960785 , 0.17254901960784313),
    (0.79607843137254897, 0.094117647058823528, 0.11372549019607843),
#    (0.6470588235294118 , 0.058823529411764705, 0.08235294117647058),
#     (0.40392156862745099, 0.0                 , 0.05098039215686274)
    )
    , 
    'LightReds'
)

greys_cm = colors.ListedColormap(
    (
    (0.85, 0.85, 0.85),
    (0.4, 0.4, 0.4),
    )
    , 
    'CustomGreys'
)

In [ ]:
def shorten_buch_label(text):
    text = text.replace('Buch', 'B.')
    text = re.sub(r'erstes', '1.', text, flags=re.IGNORECASE)
    text = re.sub(r'zweites', '2.', text, flags=re.IGNORECASE)
    text = re.sub(r'drittes', '3.', text, flags=re.IGNORECASE)
    text = re.sub(r'viertes', '4.', text, flags=re.IGNORECASE)
    text = re.sub(r'f[üu]e?nftes', '5.', text, flags=re.IGNORECASE)
    text = re.sub(r'sechstes', '6.', text, flags=re.IGNORECASE)
    text = re.sub(r'siebtes', '7.', text, flags=re.IGNORECASE)
    return text


def make_community_graph_for_snapshot(filename, country_code, weight_threshold, size_threshold, size_divisor):
    clustering = get_clustering_result(
        filename,
        country_code,
        'clustering',
         path_prefix='../',
    )
    add_community_to_graph(clustering)
    cG = quotient_graph(clustering.graph, 'community', self_loops=False, root_level=None)
    cG = make_weighted(cG)

    # Add labels
    if country_code.lower() == 'de':
        node_abks = {
            n: n.split("_")[1] 
            if data['level'] == 0 
            else n.split("_")[1] + ', ' + shorten_buch_label(" ".join(data.get("heading", '').split()[:2]))
            for n, data in clustering.graph.nodes(data=True)
        }
    else: # assuming 'us'
        node_abks = {
            # Title number plus chapter/etc. name
            n: data['law_name'].split('-')[0].split(' ')[-1] 
            if data['level'] == 0
            else f"{data['law_name'].split('-', 1)[0].split(' ', 1)[-1]}/{data['heading'].split('-', 1)[0].split(' ', 1)[-1]}".replace("&ndash;", '-')
            for n, data in G.nodes(data=True)
        }
    community_abks = {com: Counter() for com in range(len(clustering.communities))}
    for n, com in nx.get_node_attributes(clustering.graph, 'community').items():
        community_abks[com].update({node_abks[n]: clustering.graph.nodes[n]['tokens_n']})
    labels = {com: f'\n{com+1}\n{counter.most_common(1)[0][0]}' for com, counter in community_abks.items()} 
    abks = {com: counter.most_common(1)[0][0].split(',')[0] for com, counter in community_abks.items()}
    
    nx.set_node_attributes(cG, labels, name='label')
    nx.set_node_attributes(cG, abks, name='abk')


    # Filter Graph
    cG.remove_edges_from([(u,v) for u,v,w in cG.edges(data='weight') if w <= weight_threshold])
    cG = nx.subgraph(cG, [n for n, size in cG.nodes(data='tokens_n', default=0) if size >= size_threshold])
    nx.set_node_attributes(cG, {n:d/size_divisor for n,d in cG.nodes(data='tokens_n')}, name='node_size')
    return cG


def make_chapter_graph_for_snapshot(snapshot, country_code, weight_threshold, size_threshold, size_divisor):
    G = nx.read_gpickle(f'../../legal-networks-data/{country_code.lower()}/4_crossreference_graph/seqitems/{snapshot}.gpickle.gz')
    # making this quotient graph takes too much time, implementation probably inefficient
    G, _ = quotient_graph_with_merge(G, self_loops=True, merge_threshold=-1)
    # remove all nodes above the chapter level
    G.remove_nodes_from(set(e[-1] for e in G.nodes(data='parent_key')))
    G.remove_node('root')
    G = make_weighted(G)
    nx.set_node_attributes(G, {u:w for u,v,w in G.edges(data='weight') 
                                    if u == v}, name='self_references'
                              )
    if country_code.lower() == 'de':
        node_ids = {
            n: n.split("_")[1] 
            if data['level'] == 0 
            else n.split("_")[1] + ', ' + shorten_buch_label(" ".join(data.get("heading", '').split()[:2]))
            for n, data in G.nodes(data=True)
        }
        
    else: # assuming 'us'
        node_ids = {
            # Title number plus chapter/etc. name
            n: data['law_name'].split('-')[0].split(' ')[-1] 
            if data['level'] == 0
            else f"{data['law_name'].split('-', 1)[0].split(' ', 1)[-1]}/{data['heading'].split('-', 1)[0].split(' ', 1)[-1]}".replace("&ndash;", '-')
            for n, data in G.nodes(data=True)
        }
    nx.set_node_attributes(G, node_ids, name='label')
    nx.set_node_attributes(G, {node_id:node_size/size_divisor for node_id, node_size in dict(G.nodes(data='tokens_n', default=0)).items()}, 
                           name='node_size')
    nx.set_node_attributes(G, {law_name:data['self_references']/data['tokens_n'] 
                                if data.get('self_references',0) > 0 else 0.
                                for law_name, data in G.nodes(data=True)
                               }, name='internal_density'
                          )
    G.remove_edges_from([(u,v) for u,v,w in G.edges(data='weight') if w <= weight_threshold])
    G.remove_edges_from([(u,v) for u,v in G.edges() if u == v])
    G = nx.subgraph(G, [n for n, size in G.nodes(data='tokens_n', default=0) if size >= size_threshold])
    return G


def sort_edges_by_weight(G, reverse=False, filter_weight=0):
    edges_sorted = [e for e in sorted(G.edges(data=True), key=lambda tup:tup[-1]['weight'], reverse=reverse) if e[-1]['weight'] >= filter_weight]
    edgeweights_sorted = [e[-1]['weight'] for e in edges_sorted]
    return edges_sorted, edgeweights_sorted


def sort_nodes_by_size(cG, size_attribute, color_attribute):
    nodes_sorted = sorted([(c, data[size_attribute], data[color_attribute]) for c, data in cG.nodes(data=True)],
                          key=lambda tup:tup[1], reverse=True
                         )
    return zip(*nodes_sorted)


def plot_community_graph(cG, size_attribute='node_size', color_attribute='internal_density',
                         labels=None, close=False, figsize=(16,16), edge_filter_weight=0, 
                         savepath=None, size_divisor=None, edge_divisor=15, spring_layout_k=2.2, default_label_size=44, label_size_power=0.46, 
                         graycolor=False
                        ):
    assert size_divisor
    plt.rcParams['figure.figsize'] = figsize
    plt.box(False)
    if labels is None:
        labels = {node_id:node_id for node_id in cG.nodes()}
    pos = nx.spring_layout(cG, k=spring_layout_k, seed=0)
#     pos = forceatlas.forceatlas2_networkx_layout(cG, pos=None, iterations=2000)
    edges_sorted, edgeweights_sorted = sort_edges_by_weight(cG, reverse=False, filter_weight=edge_filter_weight)
    nodes_sorted, node_sizes_sorted, node_colors_sorted = sort_nodes_by_size(cG, size_attribute, color_attribute)

    min_max_row = dict(
        edge_weight_min= min(edgeweights_sorted),
        edge_weight_max= max(edgeweights_sorted),
        node_size_min= min(node_sizes_sorted)*size_divisor,
        node_size_max= max(node_sizes_sorted)*size_divisor,
        node_color_min= min(node_colors_sorted),
        node_color_max= max(node_colors_sorted)
    )
    print(min_max_row)
    
    edges_sorted_reference = [e for e in edges_sorted if e[-1].get('weight_reference', 0)]
    edges_sorted_cooccurrence = [e for e in edges_sorted if e[-1].get('weight_cooccurrence', 0) and e[0] < e[1]]
    
    edgeweights_sorted_reference = [e[-1].get('weight_reference', 0)/edge_divisor for e in edges_sorted_reference]
    edgeweights_sorted_cooccurrence = [e[-1].get('weight_cooccurrence', 0)/edge_divisor for e in edges_sorted_cooccurrence]
    
    edges_cooccurrence = nx.draw_networkx_edges(
        cG, 
        pos=pos, 
        width=edgeweights_sorted_cooccurrence,
        edgelist=edges_sorted_cooccurrence,
        edge_color="grey" if graycolor else 'lightgreen',
        arrowstyle='-',
        alpha=0.6,
#         connectionstyle='arc3,rad=0.1',
    )
    
    edges_reference = nx.draw_networkx_edges(
        cG, 
        pos=pos, 
        width=edgeweights_sorted_reference,
        edgelist=edges_sorted_reference,
        edge_color='k',
        arrowstyle='-',
        alpha=0.6,
        connectionstyle='arc3,rad=0.1',
    )
    
    if edgeweights_sorted_cooccurrence:
        edge_cooccurrence_weight_max = max(edgeweights_sorted_cooccurrence)
        for i in range(len(edges_sorted_cooccurrence)):
            edges_cooccurrence[i].set_alpha(0.1+edgeweights_sorted_cooccurrence[i]/edge_cooccurrence_weight_max*0.55)
    
    if edgeweights_sorted_reference:
        edge_reference_weight_max = max(edgeweights_sorted_reference)
        for i in range(len(edges_sorted_reference)):
            edges_reference[i].set_alpha(0.1+edgeweights_sorted_reference[i]/edge_reference_weight_max*0.55)

    nodes = nx.draw_networkx_nodes(cG, nodelist=nodes_sorted, pos=pos, node_size=node_sizes_sorted, node_color=node_colors_sorted, 
                                   cmap=greys_cm if graycolor else light_reds_cm, alpha=0.8)
    #for n in sorted([c[0] for c in cG.nodes(data='node_size')], key=lambda c:c[-1], reverse=True):
    #    nx.draw_networkx_nodes(cG, nodelist=[n], pos=pos, node_size=node_sizes[n], node_color=[node_colors[n]], cmap=cm.Reds)
    plt_labels = nx.draw_networkx_labels(cG, pos=pos, labels=labels)
    for node, label in plt_labels.items():
        size = cG.nodes[node][size_attribute]
        label.set_fontsize((size/10000)**(label_size_power)*default_label_size) # Font size 42 at size 10k
        
    plt.tight_layout()
    if close:
        plt.close()
    if savepath is not None:
        plt.savefig(savepath)
        subprocess.run(["pdfcrop", savepath, savepath], check=True)
    return min_max_row

def undirected_graph_and_sum_weights(G, weights_attr='weight'):
    H = G.to_undirected()
    weight_sums = {}
    for edge, weight in nx.get_edge_attributes(H, weights_attr).items():
        undirected_edge = tuple(sorted(edge))
        if undirected_edge in weight_sums:
            weight_sums[undirected_edge] += weight
        else:
            weight_sums[undirected_edge] = weight
    nx.set_edge_attributes(H,weight_sums, 'weight')
    return H
        

def add_betweenness_to_graph(qG):
    qGc = qG.subgraph(max(nx.connected_components(qG.to_undirected()), key=len))
    betweenness = nx.approximate_current_flow_betweenness_centrality(undirected_graph_and_sum_weights(qGc), weight='weight',solver='lu')
    nx.set_node_attributes(qG, {n: betweenness.get(n, 0.0) for n in qG.nodes}, 'betweenness')

# Hierarchisches Modell

## US - Chapter - 1994

In [ ]:
size_divisor = 100
qG = make_chapter_graph_for_snapshot(
    snapshot=1994, 
    country_code='us', 
    weight_threshold=0, 
    size_threshold=5000, 
    size_divisor=size_divisor
)
add_betweenness_to_graph(qG)

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_us_chapter_1994.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=2.6
)

## US - Chapter - 2019

In [ ]:
size_divisor = 100
qG = make_chapter_graph_for_snapshot(
    snapshot=2019, 
    country_code='us', 
    weight_threshold=0, 
    size_threshold=5000, 
    size_divisor=size_divisor
)
add_betweenness_to_graph(qG)

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_us_chapter_2019.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=2.6
)

## DE - Gesetz/Buch - 1994

In [ ]:
size_divisor = 50
qG = make_chapter_graph_for_snapshot(
    snapshot='1994-01-01', 
    country_code='de', 
    weight_threshold=0, 
    size_threshold=5000, 
    size_divisor=size_divisor
)
add_betweenness_to_graph(qG)

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_de_buch_1994.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=2.2
)
save_used_abks('meso_quotient_graph_de_buch_1994', [n.split('_')[1] for n in qG.nodes])

## DE - Gesetz/Buch - 2019

In [ ]:
size_divisor = 50
qG = make_chapter_graph_for_snapshot(
    snapshot='2019-01-01', 
    country_code='de', 
    weight_threshold=0, 
    size_threshold=5000, 
    size_divisor=size_divisor
)
add_betweenness_to_graph(qG)

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_de_buch_2019.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=2.0,
    edge_divisor=30,
)
save_used_abks('meso_quotient_graph_de_buch_2019', [n.split('_')[1] for n in qG.nodes])

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_de_buch_2019_graycolor.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=2.0,
    edge_divisor=30,
    graycolor=True
)

## DE - Commmunity - 1994 (Louvain, mixed)

In [ ]:
size_divisor = 200
qG = make_community_graph_for_snapshot(
    filename='1994-01-01_0-0_1-0_-1_o-2-0_t-paragraph_a-louvain_m0-5_s0_c1000.json', 
    country_code='de', 
    weight_threshold=0,
    size_threshold=5000, 
    size_divisor=size_divisor,
)
add_betweenness_to_graph(qG)

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_de_community_louvain_mixed_1994.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=9,
    default_label_size=36,
    label_size_power=0.15,
    edge_divisor=30
)
save_used_abks('meso_quotient_graph_de_community_louvain_mixed_1994',  {abk for n, abk in qG.nodes(data='abk') if abk})

## DE - Commmunity - 2019 (Louvain, mixed)

In [ ]:
size_divisor = 200
qG = make_community_graph_for_snapshot(
    filename='2019-01-01_0-0_1-0_-1_o-2-0_t-paragraph_a-louvain_m0-5_s0_c1000.json', 
    country_code='de', 
    weight_threshold=0,
    size_threshold=5000, 
    size_divisor=size_divisor,
)
add_betweenness_to_graph(qG)

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_de_community_louvain_mixed_2019.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=12,
    default_label_size=36,
    label_size_power=0.18,
    edge_divisor=40
)
save_used_abks('meso_quotient_graph_de_community_louvain_mixed_2019',  {abk for n, abk in qG.nodes(data='abk') if abk})

In [ ]:
row = plot_community_graph(
    qG, 
    edge_filter_weight=edge_filter_weight, 
    size_attribute='node_size', 
    labels=dict(qG.nodes(data='label')),
    savepath=f'../data_figures/meso_quotient_graph_de_community_louvain_mixed_2019_graycolor.pdf', 
    size_divisor=size_divisor,
    color_attribute='betweenness',
    spring_layout_k=12,
    default_label_size=32,
    label_size_power=0.18,
    edge_divisor=40,
    graycolor=True
)
save_used_abks('meso_quotient_graph_de_community_louvain_mixed_2019_graycolor',  {abk for n, abk in qG.nodes(data='abk') if abk})

In [ ]:
clustering = get_clustering_result(
    '2019-01-01_0-0_1-0_-1_o-2-0_t-paragraph_a-louvain_m0-5_s0_c1000.json',
    'de',
    'clustering',
     path_prefix='../',
)
add_community_to_graph(clustering)

latex =  clustering_to_community_abk_latex(clustering)
with open('../tables/meso_quotient_graph_de_community_louvain_mixed_2019.tex', 'w') as f:
    f.write(latex)
print(latex)